## 02. 전처리 로직 검증

주소 파싱, 추정평형·평형대 분류, 직거래·취소 거래 필터를 검증합니다.

In [ ]:
import sys
from pathlib import Path

PROJECT_ROOT = str(Path.cwd().resolve().parents[2])
if PROJECT_ROOT not in sys.path:
    sys.path.insert(0, PROJECT_ROOT)

from projects.part10_price_volume.dashboard.db import execute_query
from projects.part10_price_volume.dashboard.preprocessing import (
    get_region_options, get_individual_trades, get_monthly_summary, get_summary_stats
)
from src.utils.address import SIDO_SQL, SIGUNGU_SQL, EUPMYEONDONG_SQL

### 1. 주소 파싱 검증

시군구 컬럼(하나의 문자열)을 시도/시군구/읍면동으로 분리하는 SQL 로직을 검증한다.

In [ ]:
# 세종특별자치시 더블스페이스 정규화 확인
df = execute_query("""
    SELECT
        시군구 AS 원본,
        regexp_replace(시군구, '\\s+', ' ', 'g') AS 정규화,
        COUNT(*) AS cnt
    FROM 매매
    WHERE 시군구 LIKE '세종%'
    GROUP BY 시군구
    ORDER BY cnt DESC
    LIMIT 5
""")
print(df)

In [ ]:
# 시도/시군구/읍면동 분리 결과 확인 (다양한 주소 패턴)
df = execute_query(f"""
    SELECT
        시군구 AS 원본주소,
        {SIDO_SQL} AS 시도,
        {SIGUNGU_SQL} AS 시군구_parsed,
        {EUPMYEONDONG_SQL} AS 읍면동
    FROM 매매
    WHERE 시군구 IS NOT NULL
    ORDER BY RANDOM()
    LIMIT 10
""")
display(df)

In [ ]:
# 지역 계층 데이터 구조 확인
regions = get_region_options()
print(f"시도 수: {len(regions)}개")
for 시도 in sorted(regions.keys()):
    시군구_수 = len(regions[시도])
    print(f"  {시도}: {시군구_수}개 시군구")

In [ ]:
# 마포구 단지 목록 확인
마포_단지 = regions.get('서울특별시', {}).get('마포구', {})
print(f"마포구 읍면동 수: {len(마포_단지)}개")
for 읍면동, 단지목록 in list(마포_단지.items())[:5]:
    print(f"  {읍면동}: {단지목록[:3]}")

### 2. 추정평형·평형대 분류 검증

추정평형 = `FLOOR(전용면적 × 0.4)` — 소수점 이하 버림

In [ ]:
# 전용면적별 추정평형·평형대 확인 (대표 면적)
df = execute_query("""
    SELECT
        전용면적,
        CAST(FLOOR(전용면적 * 0.4) AS INTEGER) AS 추정평형,
        CASE
            WHEN FLOOR(전용면적 * 0.4) < 10 THEN '10평 미만'
            WHEN FLOOR(전용면적 * 0.4) < 20 THEN '10평대'
            WHEN FLOOR(전용면적 * 0.4) < 30 THEN '20평대'
            WHEN FLOOR(전용면적 * 0.4) < 40 THEN '30평대'
            WHEN FLOOR(전용면적 * 0.4) < 50 THEN '40평대'
            WHEN FLOOR(전용면적 * 0.4) < 60 THEN '50평대'
            ELSE '60평 이상'
        END AS 평형대,
        COUNT(*) AS 거래량
    FROM 매매
    GROUP BY 전용면적, 추정평형, 평형대
    ORDER BY 전용면적
    LIMIT 15
""")
display(df)

In [ ]:
# 평형대별 거래량 비율
df = execute_query("""
    SELECT
        CASE
            WHEN FLOOR(전용면적 * 0.4) < 10 THEN '10평 미만'
            WHEN FLOOR(전용면적 * 0.4) < 20 THEN '10평대'
            WHEN FLOOR(전용면적 * 0.4) < 30 THEN '20평대'
            WHEN FLOOR(전용면적 * 0.4) < 40 THEN '30평대'
            WHEN FLOOR(전용면적 * 0.4) < 50 THEN '40평대'
            WHEN FLOOR(전용면적 * 0.4) < 60 THEN '50평대'
            ELSE '60평 이상'
        END AS 평형대,
        COUNT(*) AS 거래량,
        ROUND(COUNT(*) * 100.0 / SUM(COUNT(*)) OVER (), 1) AS 비율
    FROM 매매
    GROUP BY 평형대
    ORDER BY MIN(전용면적)
""")
display(df)

### 3. 직거래·취소 거래 필터 검증

In [ ]:
# 직거래 제외 전후 비교 (마포구, 2025년)
df_all = execute_query("""
    SELECT COUNT(*) AS cnt FROM 매매
    WHERE 시군구 LIKE '서울특별시 마포구%'
      AND 계약년월 >= 202500
""")
df_filtered = execute_query("""
    SELECT COUNT(*) AS cnt FROM 매매
    WHERE 시군구 LIKE '서울특별시 마포구%'
      AND 계약년월 >= 202500
      AND (거래유형 IS NULL OR 거래유형 != '직거래')
      AND (해제사유발생일 IS NULL OR 해제사유발생일 IN ('None', '-', ''))
""")
total = df_all.iloc[0]['cnt']
filtered = df_filtered.iloc[0]['cnt']
print(f"전체: {total:,}건")
print(f"필터 후: {filtered:,}건 (제외: {total - filtered:,}건, {(total - filtered)/total*100:.1f}%)")

### 4. 개별 거래 조회 — 마포래미안푸르지오 84㎡

대시보드에서 사용하는 `get_individual_trades()`를 직접 호출해본다.

In [ ]:
data = get_individual_trades(
    시도='서울특별시',
    시군구='마포구',
    단지=['마포래미안푸르지오'],
    평형대=['30평대'],
    시작년월=202000,
    종료년월=202509,
)
print(f"매매 건수: {len(data.get('매매', []))},  전세 건수: {len(data.get('전세', []))}")
display(data.get('매매', []).tail())

### 5. 월별 집계 조회 — 마포래미안푸르지오

`get_monthly_summary()`로 월별 중위가와 거래량을 확인한다.

In [ ]:
monthly = get_monthly_summary(
    시도='서울특별시',
    시군구='마포구',
    단지=['마포래미안푸르지오'],
    평형대=['30평대'],
    시작년월=202000,
    종료년월=202509,
)
df_monthly = monthly.get('매매', [])
print(f"월별 데이터 {len(df_monthly)}개월")
display(df_monthly.tail(10))

In [ ]:
# 요약 통계
stats = get_summary_stats(
    시도='서울특별시',
    시군구='마포구',
    단지=['마포래미안푸르지오'],
    평형대=['30평대'],
    시작년월=202000,
    종료년월=202509,
)
print(f"총 거래건수: {stats['총거래건수']:,}건")
print(f"최고가: {stats['최고가_억']}억  최저가: {stats['최저가_억']}억")
print("\n최근 거래 3건:")
display(stats['최근거래'])